# New Baselines & Structural Comparison — Twitch Networks

This notebook addresses reviewer concerns **W3**, **W4**, and **W5**:

- **W5 (Missing baselines):** Adds **Stochastic Block Model (SBM)**, **Configuration Model**, and **Chung-Lu** as baselines alongside ER, BA, WS.
- **W4 (Non-spectral metrics):** Directly compares degree distributions (KS test), clustering, path lengths, modularity, assortativity, and triangle counts.
- **W3 (Circular spectral evaluation):** Provides non-spectral evidence that the LG model captures structural properties beyond the spectrum.

**Datasets:** All six Twitch gamer networks (DE, EN, ES, FR, PT, RU).

In [1]:
import sys, os, warnings, pickle
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import ks_2samp
from collections import defaultdict

sys.path.insert(0, os.path.abspath('../..'))
from src.logit_graph.simulation import (
    LogitGraphFitter, LogitGraphSimulation,
    estimate_sigma_only, calculate_graph_attributes,
)
from src.logit_graph import gic

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
mpl.rcParams.update({
    'font.family': 'serif', 'font.size': 14,
    'axes.titlesize': 16, 'axes.labelsize': 14,
    'figure.figsize': (14, 6),
})

SEED = 42
np.random.seed(SEED)
OUT_DIR = 'runs'
os.makedirs(OUT_DIR, exist_ok=True)
print('Setup complete.')

Setup complete.


## 1. Helper Functions

In [2]:
def compute_structural_features(G, max_path_nodes=3000, seed=42):
    """Compute non-spectral structural metrics for a graph."""
    feats = {}
    n = G.number_of_nodes()
    m = G.number_of_edges()
    feats['nodes'] = n
    feats['edges'] = m
    feats['density'] = nx.density(G)

    degrees = [d for _, d in G.degree()]
    feats['avg_degree'] = np.mean(degrees)
    feats['std_degree'] = np.std(degrees)
    feats['max_degree'] = max(degrees) if degrees else 0
    feats['degree_skewness'] = float(pd.Series(degrees).skew()) if len(degrees) > 2 else 0.0

    feats['avg_clustering'] = nx.average_clustering(G)
    feats['transitivity'] = nx.transitivity(G)

    try:
        feats['assortativity'] = nx.degree_assortativity_coefficient(G)
    except Exception:
        feats['assortativity'] = np.nan

    feats['triangles'] = sum(nx.triangles(G).values()) // 3

    components = list(nx.connected_components(G))
    feats['num_components'] = len(components)
    largest_cc = max(components, key=len)
    feats['largest_cc_fraction'] = len(largest_cc) / n

    if len(largest_cc) > 1 and len(largest_cc) <= max_path_nodes:
        sub = G.subgraph(largest_cc)
        feats['avg_path_length'] = nx.average_shortest_path_length(sub)
        feats['diameter'] = nx.diameter(sub)
    else:
        feats['avg_path_length'] = np.nan
        feats['diameter'] = np.nan

    try:
        comms = nx.community.louvain_communities(G, seed=seed)
        feats['modularity'] = nx.community.modularity(G, comms)
        feats['n_communities'] = len(comms)
    except Exception:
        feats['modularity'] = np.nan
        feats['n_communities'] = np.nan

    return feats


def degree_ks_stat(G_real, G_model):
    """KS statistic between degree distributions."""
    d_real = [d for _, d in G_real.degree()]
    d_model = [d for _, d in G_model.degree()]
    stat, _ = ks_2samp(d_real, d_model)
    return stat


def compute_gic_value(G_real, G_model, n_params):
    """Compute GIC (spectral distance + complexity penalty)."""
    try:
        calc = gic.GraphInformationCriterion(
            graph=G_real, model='LG', log_graph=G_model, dist='KL'
        )
        return calc.calculate_gic(n_params=n_params)
    except Exception as e:
        print(f'    GIC error: {e}')
        return np.nan

print('Structural feature and metric helpers defined.')

Structural feature and metric helpers defined.


In [3]:
# ---------- New baseline generators ----------

def generate_sbm(G_real, seed=None):
    """SBM fitted via Louvain community detection on G_real."""
    comms = list(nx.community.louvain_communities(G_real, seed=seed))
    sizes = [len(c) for c in comms]
    k = len(comms)

    comm_nodes = [sorted(c) for c in comms]
    node_to_comm = {}
    for ci, nodes in enumerate(comm_nodes):
        for v in nodes:
            node_to_comm[v] = ci

    p_matrix = np.zeros((k, k))
    for ci in range(k):
        for cj in range(ci, k):
            ni, nj = comm_nodes[ci], comm_nodes[cj]
            if ci == cj:
                possible = len(ni) * (len(ni) - 1) / 2
                actual = sum(1 for u in ni for v in ni if u < v and G_real.has_edge(u, v))
            else:
                possible = len(ni) * len(nj)
                actual = sum(1 for u in ni for v in nj if G_real.has_edge(u, v))
            p_matrix[ci, cj] = actual / possible if possible > 0 else 0
            p_matrix[cj, ci] = p_matrix[ci, cj]

    sbm = nx.stochastic_block_model(sizes, p_matrix.tolist(), seed=seed)
    G_out = nx.Graph()
    G_out.add_nodes_from(range(sum(sizes)))
    G_out.add_edges_from(sbm.edges())
    n_params = k * (k + 1) // 2
    return G_out, n_params


def generate_config_model(G_real, seed=None):
    """Configuration model preserving the exact degree sequence."""
    deg_seq = [d for _, d in G_real.degree()]
    if sum(deg_seq) % 2 != 0:
        deg_seq[0] += 1
    G_cm = nx.configuration_model(deg_seq, seed=seed)
    G_cm = nx.Graph(G_cm)
    G_cm.remove_edges_from(nx.selfloop_edges(G_cm))
    return G_cm, 1


def generate_chung_lu(G_real, seed=None):
    """Chung-Lu model matching expected degree sequence."""
    deg_seq = [d for _, d in G_real.degree()]
    G_cl = nx.expected_degree_graph(deg_seq, seed=seed, selfloops=False)
    return G_cl, 1


# ---------- Existing baseline generators ----------

def generate_er(G_real, seed=None):
    return nx.erdos_renyi_graph(G_real.number_of_nodes(), nx.density(G_real), seed=seed), 1


def generate_ba(G_real, seed=None):
    avg_k = 2 * G_real.number_of_edges() / G_real.number_of_nodes()
    m = max(1, int(round(avg_k / 2)))
    return nx.barabasi_albert_graph(G_real.number_of_nodes(), m, seed=seed), 1


def generate_ws(G_real, seed=None):
    n = G_real.number_of_nodes()
    avg_k = 2 * G_real.number_of_edges() / n
    k = max(2, int(round(avg_k)))
    if k % 2 != 0:
        k += 1
    k = min(k, n - 1)
    return nx.watts_strogatz_graph(n, k, 0.1, seed=seed), 2


def estimate_lg_params(adj, d):
    """Estimate sigma AND beta jointly (fix_beta=False).

    With fix_beta=True the coefficient of S_i+S_j is forced to 1, but the
    raw degree-sum features can be so large that the logistic function
    saturates.  Estimating beta freely yields a small scaling factor and
    a sigma in the [-10, -2] range consistent with the paper.
    """
    from src.logit_graph.logit_estimator import LogitRegEstimator
    est = LogitRegEstimator(adj, d=d, verbose=False)
    features, labels = est.get_features_labels()
    _, params, _ = est.estimate_parameters(
        features=features, labels=labels, fix_beta=False, alpha=0,
    )
    sigma, beta = float(params[0]), float(params[1])
    return sigma, beta


def simulate_lg(n, d, sigma, beta=1.0, max_iter=15000, patience=2000, er_p=0.05, seed=None):
    """Generate an LG graph via MCMC simulation."""
    sim = LogitGraphSimulation(
        n=n, d=d, sigma=sigma, beta=beta,
        er_p=er_p,
        n_iteration=max_iter, warm_up=500, patience=patience,
        check_interval=50, verbose=False,
    )
    sim.simulate()
    return sim.simulated_graph, 1

print('Model generators defined.')

Model generators defined.


## 2. Load Twitch Networks

In [4]:
DATA_DIR = '../../data/twitch/graphs_processed'

TWITCH_MAP = {
    'Twitch (DE)': 'DE_graph.edges',
    'Twitch (EN)': 'ENGB_graph.edges',
    'Twitch (ES)': 'ES_graph.edges',
    'Twitch (FR)': 'FR_graph.edges',
    'Twitch (PT)': 'PTBR_graph.edges',
    'Twitch (RU)': 'RU_graph.edges',
}

datasets = {}
for name, fname in TWITCH_MAP.items():
    path = os.path.join(DATA_DIR, fname)
    G = nx.read_edgelist(path, nodetype=int)
    G = G.to_undirected()
    G.remove_edges_from(nx.selfloop_edges(G))
    datasets[name] = G
    print(f'{name:16s}  n={G.number_of_nodes():>6,}  m={G.number_of_edges():>8,}')

print(f'\nLoaded {len(datasets)} networks.')

Twitch (DE)       n= 9,498  m= 153,138
Twitch (EN)       n= 7,126  m=  35,324
Twitch (ES)       n= 4,648  m=  59,382
Twitch (FR)       n= 6,549  m= 112,666
Twitch (PT)       n= 1,912  m=  31,299
Twitch (RU)       n= 4,385  m=  37,304

Loaded 6 networks.


## 3. Configuration

Adjust `N_SAMPLES` (instances per model for averaging) and `LG_MAX_ITER` as needed.  
The LG simulation is the bottleneck; set `SKIP_LG = True` to skip it and only evaluate the fast baselines.

In [5]:
N_SAMPLES   = 5      # model instances for averaging
LG_MAX_ITER = 15000  # MCMC iterations for LG (increase for better convergence)
LG_PATIENCE = 2000
SKIP_LG     = False  # set True to skip LG generation (fast run)

## 4. Fit All Models & Compute Metrics

For each Twitch network we:
1. Estimate LG $\sigma$ via logistic regression (fast) and generate via MCMC.
2. Generate ER, BA, WS graphs (existing baselines).
3. Generate **SBM**, **Configuration Model**, **Chung-Lu** graphs (new baselines).
4. Compute **GIC** (spectral) and **non-spectral structural metrics** for every model.

In [ ]:
MODEL_SPECS = {
    'ER':       {'gen': generate_er,           'label': 'Erd\u0151s-R\u00e9nyi'},
    'BA':       {'gen': generate_ba,           'label': 'Barab\u00e1si-Albert'},
    'WS':       {'gen': generate_ws,           'label': 'Watts-Strogatz'},
    'SBM':      {'gen': generate_sbm,          'label': 'Stochastic Block Model'},
    'Config':   {'gen': generate_config_model, 'label': 'Configuration Model'},
    'Chung-Lu': {'gen': generate_chung_lu,     'label': 'Chung-Lu'},
}

all_rows = []
degree_cdfs = {}  # for plotting later

for ds_name, G_real in datasets.items():
    print(f"\n{'='*65}")
    print(f"  {ds_name}  (n={G_real.number_of_nodes():,}, m={G_real.number_of_edges():,})")
    print(f"{'='*65}")

    real_feats = compute_structural_features(G_real)
    real_feats.update({'model': 'Real', 'dataset': ds_name, 'gic': 0.0, 'gic_std': 0.0, 'ks_degree': 0.0})
    all_rows.append(real_feats)
    degree_cdfs.setdefault(ds_name, {})['Real'] = sorted([d for _, d in G_real.degree()])

    # --- LG model ---
    if not SKIP_LG:
        print('  [LG] Estimating sigma + beta (fix_beta=False)...')
        adj = nx.to_numpy_array(G_real)
        best_d, best_sigma, best_beta = None, None, None
        for d_cand in [0, 1, 2]:
            s, b = estimate_lg_params(adj, d=d_cand)
            print(f'        d={d_cand}  sigma={s:.4f}  beta={b:.6f}')
            if best_sigma is None or abs(s) < abs(best_sigma):
                best_sigma, best_beta, best_d = s, b, d_cand
        del adj

        er_p = nx.density(G_real)
        print(f'  [LG] Generating {N_SAMPLES} graphs with d={best_d}, sigma={best_sigma:.4f}, beta={best_beta:.6f}...')
        lg_gics, lg_kss, lg_feats = [], [], []
        for s_i in range(N_SAMPLES):
            G_lg, _ = simulate_lg(
                G_real.number_of_nodes(), best_d, best_sigma,
                beta=best_beta, max_iter=LG_MAX_ITER,
                patience=LG_PATIENCE, er_p=er_p,
            )
            if G_lg is not None:
                lg_gics.append(compute_gic_value(G_real, G_lg, n_params=1))
                lg_kss.append(degree_ks_stat(G_real, G_lg))
                lg_feats.append(compute_structural_features(G_lg))
                if s_i == 0:
                    degree_cdfs[ds_name]['LG'] = sorted([d for _, d in G_lg.degree()])

        if lg_feats:
            avg = {k: np.nanmean([f[k] for f in lg_feats]) for k in lg_feats[0]}
            avg.update({
                'model': 'LG', 'dataset': ds_name,
                'gic': np.nanmean(lg_gics), 'gic_std': np.nanstd(lg_gics),
                'ks_degree': np.nanmean(lg_kss),
            })
            all_rows.append(avg)

    # --- Other models ---
    for model_name, spec in MODEL_SPECS.items():
        print(f'  [{model_name}] Generating {N_SAMPLES} instances...')
        gics, kss, feats_list = [], [], []
        for s_i in range(N_SAMPLES):
            G_m, n_params = spec['gen'](G_real, seed=SEED + s_i)
            gics.append(compute_gic_value(G_real, G_m, n_params))
            kss.append(degree_ks_stat(G_real, G_m))
            feats_list.append(compute_structural_features(G_m))
            if s_i == 0:
                degree_cdfs[ds_name][model_name] = sorted([d for _, d in G_m.degree()])

        avg = {k: np.nanmean([f[k] for f in feats_list]) for k in feats_list[0]}
        avg.update({
            'model': model_name, 'dataset': ds_name,
            'gic': np.nanmean(gics), 'gic_std': np.nanstd(gics),
            'ks_degree': np.nanmean(kss),
        })
        all_rows.append(avg)

df = pd.DataFrame(all_rows)
print(f'\nDone. Collected {len(df)} rows ({len(df["dataset"].unique())} datasets).')


  Twitch (DE)  (n=9,498, m=153,138)
  [LG] Estimating sigma...
        d=0  sigma=-42.4719
        d=1  sigma=-10922.5000
        d=2  sigma=-423347.0740
  [LG] Generating 5 graphs with d=0, sigma=-42.4719...


Generating graph:   0%|          | 0/15000 [00:00<?, ?it/s]

## 5. Results

### 5.1 GIC Comparison (Spectral — Table 3 Extension)

In [ ]:
pivot_gic = df.pivot_table(index='dataset', columns='model', values='gic')
model_order = ['LG', 'ER', 'BA', 'WS', 'SBM', 'Config', 'Chung-Lu', 'Real']
model_order = [m for m in model_order if m in pivot_gic.columns]
pivot_gic = pivot_gic[model_order]

def highlight_min(row):
    non_real = row.drop('Real', errors='ignore')
    is_min = non_real == non_real.min()
    return ['font-weight: bold' if v else '' for v in row.index.isin(is_min[is_min].index)]

print('GIC (lower = better spectral fit):')
display(pivot_gic.style.apply(highlight_min, axis=1).format('{:.4f}', na_rep='-'))

### 5.2 Degree-Distribution KS Statistic (Non-Spectral)

In [ ]:
pivot_ks = df.pivot_table(index='dataset', columns='model', values='ks_degree')
pivot_ks = pivot_ks[[m for m in model_order if m in pivot_ks.columns]]

print('Degree-distribution KS statistic (lower = better match):')
display(pivot_ks.style.apply(highlight_min, axis=1).format('{:.4f}', na_rep='-'))

### 5.3 Structural Metrics Comparison (Non-Spectral)

Key metrics compared: clustering, transitivity, assortativity, modularity, triangles.

In [ ]:
STRUCT_COLS = ['avg_clustering', 'transitivity', 'assortativity', 'modularity', 'triangles', 'n_communities']

for ds_name in datasets:
    sub = df[df['dataset'] == ds_name][['model'] + STRUCT_COLS].set_index('model')
    sub = sub.reindex([m for m in model_order if m in sub.index])
    print(f'\n--- {ds_name} ---')
    display(sub.style.format('{:.4f}', na_rep='-'))

### 5.4 Absolute Deviation from Real Network

For each metric, compute |model_value - real_value| to see which model is closest.

In [ ]:
deviation_rows = []
for ds_name in datasets:
    sub = df[df['dataset'] == ds_name].set_index('model')
    real = sub.loc['Real']
    for model in sub.index:
        if model == 'Real':
            continue
        row = {'dataset': ds_name, 'model': model}
        for col in STRUCT_COLS:
            r_val, m_val = real[col], sub.loc[model, col]
            if pd.notna(r_val) and pd.notna(m_val) and r_val != 0:
                row[col] = abs(m_val - r_val) / abs(r_val)
            else:
                row[col] = np.nan
        deviation_rows.append(row)

df_dev = pd.DataFrame(deviation_rows)

print('Mean relative deviation per model (lower = closer to real, across all datasets):')
mean_dev = df_dev.groupby('model')[STRUCT_COLS].mean()
mean_dev = mean_dev.reindex([m for m in model_order if m in mean_dev.index])
display(mean_dev.style.format('{:.4f}', na_rep='-').background_gradient(cmap='RdYlGn_r', axis=None))

## 6. Visualizations

In [ ]:
ds_names = list(datasets.keys())
models_to_plot = [m for m in model_order if m != 'Real']
n_ds = len(ds_names)

fig, axes = plt.subplots(1, n_ds, figsize=(4 * n_ds, 5), sharey=False)
if n_ds == 1:
    axes = [axes]

colors = plt.cm.Set2(np.linspace(0, 1, len(models_to_plot)))
color_map = dict(zip(models_to_plot, colors))

for ax, ds_name in zip(axes, ds_names):
    sub = df[(df['dataset'] == ds_name) & (df['model'] != 'Real')].set_index('model')
    sub = sub.reindex([m for m in models_to_plot if m in sub.index])
    bars = ax.bar(
        range(len(sub)), sub['gic'],
        yerr=sub.get('gic_std', 0), capsize=3,
        color=[color_map[m] for m in sub.index], edgecolor='black', linewidth=0.5
    )
    ax.set_xticks(range(len(sub)))
    ax.set_xticklabels(sub.index, rotation=45, ha='right', fontsize=10)
    ax.set_title(ds_name, fontsize=13)
    ax.set_ylabel('GIC' if ax == axes[0] else '')

fig.suptitle('GIC Comparison — All Baselines (lower = better)', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'gic_all_baselines_twitch.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, ds_name in enumerate(ds_names):
    ax = axes[idx]
    cdfs = degree_cdfs.get(ds_name, {})
    for model_name, degs in cdfs.items():
        sorted_d = np.sort(degs)
        cdf = np.arange(1, len(sorted_d) + 1) / len(sorted_d)
        lw = 2.5 if model_name == 'Real' else 1.2
        ls = '-' if model_name in ('Real', 'LG', 'SBM', 'Config', 'Chung-Lu') else '--'
        ax.plot(sorted_d, 1 - cdf, label=model_name, linewidth=lw, linestyle=ls)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(ds_name)
    ax.set_xlabel('Degree $k$')
    ax.set_ylabel('$P(K \\geq k)$' if idx % 3 == 0 else '')
    ax.legend(fontsize=7, loc='lower left')

fig.suptitle('Complementary CDF of Degree Distributions', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'degree_ccdf_twitch.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

mean_dev_plot = mean_dev.copy()
mean_dev_plot.columns = [c.replace('_', ' ').title() for c in mean_dev_plot.columns]
im = ax.imshow(mean_dev_plot.values, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=2)

ax.set_xticks(range(len(mean_dev_plot.columns)))
ax.set_xticklabels(mean_dev_plot.columns, rotation=45, ha='right')
ax.set_yticks(range(len(mean_dev_plot.index)))
ax.set_yticklabels(mean_dev_plot.index)

for i in range(len(mean_dev_plot.index)):
    for j in range(len(mean_dev_plot.columns)):
        val = mean_dev_plot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9,
                    color='white' if val > 1.0 else 'black')

plt.colorbar(im, ax=ax, label='Mean Relative Deviation from Real')
ax.set_title('Structural Deviation Heatmap (averaged across Twitch networks)')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'structural_deviation_heatmap_twitch.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
rank_rows = []
for ds_name in datasets:
    sub = df[(df['dataset'] == ds_name) & (df['model'] != 'Real')].copy()
    sub['gic_rank'] = sub['gic'].rank()
    sub['ks_rank'] = sub['ks_degree'].rank()
    for _, r in sub.iterrows():
        rank_rows.append({
            'dataset': ds_name, 'model': r['model'],
            'gic_rank': r['gic_rank'], 'ks_rank': r['ks_rank'],
        })

df_ranks = pd.DataFrame(rank_rows)

print('Mean Rank across Twitch networks (1 = best):')
mean_ranks = df_ranks.groupby('model')[['gic_rank', 'ks_rank']].mean()
mean_ranks.columns = ['GIC Rank (spectral)', 'KS Rank (degree dist.)']
mean_ranks = mean_ranks.sort_values('GIC Rank (spectral)')
display(mean_ranks.style.format('{:.2f}'))

## 7. Save Results

In [ ]:
df.to_csv(os.path.join(OUT_DIR, 'twitch_all_baselines_results.csv'), index=False)
df_dev.to_csv(os.path.join(OUT_DIR, 'twitch_structural_deviation.csv'), index=False)
df_ranks.to_csv(os.path.join(OUT_DIR, 'twitch_model_ranks.csv'), index=False)

with open(os.path.join(OUT_DIR, 'twitch_degree_cdfs.pkl'), 'wb') as f:
    pickle.dump(degree_cdfs, f)

print('All results saved to:', OUT_DIR)